# Reusable EDA / Visualization Template

Point this at a new event-log-style dataset (trips, orders, check-ins, sensor
readings) and work through it top to bottom. Everything in `ALL_CAPS` or
`<angle brackets>` is a placeholder — fill it in for your dataset, then delete
this note.

Pairs with `EDA_Viz_Strategy_Playbook.ipynb` for the reasoning behind each step.

## 0. Parameters — edit these for your dataset

In [ ]:
# --- Fill these in for your dataset, then run every cell below unchanged ---
DATA_PATH        <- "<path/to/your_data.csv>"
ID_COL           <- "<row identifier column, or NULL if none>"

TIME_COL         <- "<timestamp column, e.g. 'starttime'>"
TIME_FORMAT_FN   <- ymd_hms   # swap for mdy_hms / dmy_hms / etc. if needed

DURATION_COL     <- "<numeric duration/quantity column, e.g. 'tripduration'>"
DURATION_UNIT    <- "<units, e.g. 'seconds'>"

FROM_LAT_COL     <- "<start latitude column, or NA if not spatial>"
FROM_LON_COL     <- "<start longitude column, or NA>"
TO_LAT_COL       <- "<end latitude column, or NA>"
TO_LON_COL       <- "<end longitude column, or NA>"

FROM_NODE_COL    <- "<start location/name/id column for flow analysis, or NA>"
TO_NODE_COL      <- "<end location/name/id column for flow analysis, or NA>"

DEMO_YEAR_COL    <- "<birth year or similar column, or NA>"
REFERENCE_YEAR   <- 2020   # the year the data was collected -- NOT Sys.Date()

CATEGORY_COL     <- "<a categorical grouping column, e.g. usertype/gender/region>"
SUBSET_FILTER_MAX <- Inf   # e.g. 900 to mimic the Citi Bike 15-min subset; Inf = no cap

## 1. Setup

In [ ]:
library(dplyr)
library(ggplot2)
library(lubridate)
# library(geosphere)   # uncomment if FROM/TO lat-lon columns are set

## 2. Load & audit

In [ ]:
raw_data <- read.csv(DATA_PATH)
str(raw_data)

In [ ]:
colSums(is.na(raw_data))

In [ ]:
# Range-check your key numeric column before trusting it
range(raw_data[[DURATION_COL]], na.rm = TRUE)
sum(raw_data[[DURATION_COL]] <= 0, na.rm = TRUE)

**Audit notes:** *(replace this text — what did you find, what will you filter and why)*

## 3. Subset (optional, for speed while iterating)

In [ ]:
work_data <- raw_data
if (is.finite(SUBSET_FILTER_MAX)) {
  work_data <- work_data %>% filter(.data[[DURATION_COL]] < SUBSET_FILTER_MAX)
}
head(work_data)

## 4. Spatial heat map (skip if not applicable)

In [ ]:
if (!is.na(FROM_LAT_COL) && !is.na(FROM_LON_COL)) {
  ggplot(work_data, aes(x = .data[[FROM_LON_COL]], y = .data[[FROM_LAT_COL]])) +
    geom_bin2d(binwidth = c(0.001, 0.001)) +
    labs(title = "Density of starting locations", x = "Longitude", y = "Latitude") +
    theme(plot.title = element_text(hjust = 0.5))
}

## 5. Derived features: age, distance, speed (adapt as needed)

In [ ]:
if (!is.na(DEMO_YEAR_COL)) {
  work_data <- work_data %>% mutate(age = REFERENCE_YEAR - .data[[DEMO_YEAR_COL]])
}

if (!is.na(FROM_LAT_COL) && !is.na(TO_LAT_COL)) {
  library(geosphere)
  from_pts <- work_data %>% select(all_of(c(FROM_LON_COL, FROM_LAT_COL)))
  to_pts   <- work_data %>% select(all_of(c(TO_LON_COL, TO_LAT_COL)))
  work_data <- work_data %>% mutate(distance = distHaversine(from_pts, to_pts))
  work_data <- work_data %>% mutate(speed = distance / .data[[DURATION_COL]])
}

head(work_data)

## 6. Time features (skip if no timestamp)

In [ ]:
if (!is.na(TIME_COL)) {
  work_data <- work_data %>%
    mutate(
      parsed_time = TIME_FORMAT_FN(.data[[TIME_COL]]),
      hour        = hour(parsed_time),
      weekday     = wday(parsed_time, label = TRUE),
      is_weekend  = weekday %in% c("Sat", "Sun")
    )
}
head(work_data %>% select(any_of(c(TIME_COL, "parsed_time", "hour", "weekday", "is_weekend"))))

## 7. Primary metric by one grouping variable

In [ ]:
# Swap `age` and `speed` for whatever your primary numeric metric and grouping
# variable are.
by_one_group <- work_data %>% group_by(age) %>% summarize(mean_metric = mean(speed, na.rm = TRUE))

by_one_group %>% ggplot(aes(x = age, y = mean_metric)) + geom_line() +
  labs(title = "Metric by group", x = "Group", y = "Mean metric") +
  theme(plot.title = element_text(hjust = 0.5))

## 8. Cross-cut with a second (categorical) variable

In [ ]:
by_two_groups <- work_data %>%
  mutate(cat = as.factor(.data[[CATEGORY_COL]])) %>%
  group_by(age, cat) %>% summarize(mean_metric = mean(speed, na.rm = TRUE))

by_two_groups %>% ggplot(aes(x = age, y = mean_metric, color = cat)) + geom_line() +
  labs(title = "Metric by group and category", x = "Group", y = "Mean metric", color = CATEGORY_COL) +
  theme(plot.title = element_text(hjust = 0.5))

## 9. Volume distribution (stacked bar)

In [ ]:
counts <- work_data %>% mutate(cat = as.factor(.data[[CATEGORY_COL]])) %>%
  group_by(age, cat) %>% tally()

counts %>% ggplot(aes(x = age, y = n, fill = cat)) + geom_col() +
  labs(title = "Volume by group and category", x = "Group", y = "Count", fill = CATEGORY_COL) +
  theme(plot.title = element_text(hjust = 0.5))

## 10. Temporal load

In [ ]:
if (!is.na(TIME_COL)) {
  work_data %>% group_by(hour) %>% tally() %>%
    ggplot(aes(x = hour, y = n)) + geom_line() +
    labs(title = "Volume by hour of day", x = "Hour", y = "Count") +
    theme(plot.title = element_text(hjust = 0.5))
}

## 11. Node flow / imbalance (skip if not applicable)

In [ ]:
if (!is.na(FROM_NODE_COL) && !is.na(TO_NODE_COL)) {
  out_counts <- work_data %>% group_by(.data[[FROM_NODE_COL]]) %>% tally(name = "outflow")
  in_counts  <- work_data %>% group_by(.data[[TO_NODE_COL]]) %>% tally(name = "inflow")

  node_flow <- full_join(out_counts, in_counts, by = setNames(TO_NODE_COL, FROM_NODE_COL)) %>%
    mutate(across(c(outflow, inflow), ~replace(., is.na(.), 0)),
           net_flow = inflow - outflow)

  head(node_flow %>% arrange(net_flow))
}

## 12. Takeaway memo

1. **Headline finding:** *(one sentence)*
2. **Supporting evidence:** *(2-3 numbers/charts above)*
3. **Biggest assumption / caveat:** *(what would most change the conclusion if wrong)*
4. **Next step:** *(what data or analysis you'd want next)*